# Frontieres-11 — Abstraire à dette bornée : certifier un trajet sans le calculer

**Navigation** : [Version Python](Frontieres-11-Abstraction-Dette-ShortestPath-Python.ipynb) · [Partie 5](README.md) · [Search](../README.md)

**Objectifs** : construire une borne abstraite admissible, auditer un contre-témoin, relever une route concrète et mesurer la dette de plusieurs échelles. **Prérequis** : Dijkstra et heuristiques admissibles. **Durée estimée** : 55 minutes.

Lignée bibliographique reprise du Python versionné, sans nouvelle lecture primaire revendiquée : bases de motifs, Culberson et Schaeffer (1998), puis Holte, Grahn, Hansen et Zilberstein (2005). Compresser une carte permet de résoudre un petit problème ; la garantie ne voyage que si chaque quantité transportée reste un minorant. Le même geste est étudié pour les jeux dans [Metajeux-03](../../GameTheory/Metajeux/Metajeux-03-Abstraction-a-Dette-Python.ipynb).

La BCL implémente ici les algorithmes qui constituent l'objet pédagogique du source : aucun moteur externe de référence n'est omis. Les RNG .NET et Python diffèrent ; les cartes et valeurs mesurées diffèrent, avec les mêmes dimensions, graines et budgets.

In [1]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text;
using System.Net;
using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.CurrentCulture=CultureInfo.InvariantCulture;
record Arete(int U,int V,double W);
class Graphe:Dictionary<int,List<(int V,double W)>> {}
class Carte {
 public int H,W;public int[] Couts;public Graphe G=new();public List<Arete> Aretes=new();
 public Carte(int h,int w,int[] couts){if(h<=0||w<=0||couts.Length!=h*w||couts.Any(c=>c<=0))throw new ArgumentException("Carte positive et dimensions exactes requises");H=h;W=w;Couts=couts.ToArray();for(int u=0;u<h*w;u++)G[u]=new();for(int r=0;r<h;r++)for(int c=0;c<w;c++){int u=r*w+c;foreach(var rc in new[]{(r,c+1),(r+1,c)})if(rc.Item1<h&&rc.Item2<w){int v=rc.Item1*w+rc.Item2;double cost=Math.Max(couts[u],couts[v]);G[u].Add((v,cost));G[v].Add((u,cost));Aretes.Add(new(u,v,cost));}}}
}
record Dessin(string Content);Formatter.Register<Dessin>((v,w)=>w.Write(v.Content),"text/html");
Console.WriteLine($".NET {Environment.Version} ; graphes et algorithmes BCL ; SVG autonomes.");

.NET 10.0.14 ; graphes et algorithmes BCL ; SVG autonomes.


## 1. Une carte pondérée

Une grille 24×24 contient routes/coût 1, plaines/coût 3 et forêts/coût 7 dans les proportions 2:5:3. Un pas paie le maximum des terrains de ses deux extrémités. Les mouvements sont horizontaux/verticaux ; S et T occupent des coins opposés. La figure est un vrai SVG opaque.

In [2]:
#nullable enable
var aleaCarte=new Random(20261015);int[] couts=Enumerable.Range(0,24*24).Select(_=>{double u=aleaCarte.NextDouble();return u<.2?1:u<.7?3:7;}).ToArray();var carte=new Carte(24,24,couts);int depart=0,but=575;
string Nom(int u,Carte m)=>$"({u/m.W},{u%m.W})";
var carteSvg=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='580' height='570' viewBox='0 0 580 570'><rect width='580' height='570' fill='white'/><text x='290' y='25' text-anchor='middle'>Carte 24×24 — coût du pas : maximum des deux terrains</text>");
for(int r=0;r<24;r++)for(int c=0;c<24;c++){string couleur=carte.Couts[r*24+c]==1?"#fff5b1":carte.Couts[r*24+c]==3?"#a1d99b":"#238b45";carteSvg.Append($"<rect x='{55+c*18}' y='{50+r*18}' width='18' height='18' fill='{couleur}' stroke='white' stroke-width='.25'/>");}for(int t=0;t<24;t+=4){carteSvg.Append($"<text x='{64+t*18}' y='500' text-anchor='middle' font-size='12'>{t}</text><text x='45' y='{63+t*18}' text-anchor='end' font-size='12'>{t}</text>");}carteSvg.Append("<circle cx='64' cy='59' r='6' fill='black'/><text x='78' y='63' font-size='12'>S</text><circle cx='478' cy='473' r='6' fill='red'/><text x='490' y='477' font-size='12'>T</text><text x='270' y='522' text-anchor='middle'>colonne</text><text transform='translate(17 265) rotate(-90)' text-anchor='middle'>rangée</text>");string[] labels={"route 1","plaine 3","forêt 7"};string[] couleurs={"#fff5b1","#a1d99b","#238b45"};for(int i=0;i<3;i++)carteSvg.Append($"<rect x='{110+i*135}' y='542' width='16' height='16' fill='{couleurs[i]}'/><text x='{132+i*135}' y='555' font-size='13'>{labels[i]}</text>");carteSvg.Append("</svg>");display(new Dessin(carteSvg.ToString()));Console.WriteLine($"Carte {carte.H}×{carte.W}, {carte.Aretes.Count} arêtes ; coûts 1/3/7 ; graine 20261015, RNG .NET.");

Carte 24×24 — coût du pas : maximum des deux terrains 0 0 4 4 8 8 12 12 16 16 20 20 S T colonne rangée route 1 plaine 3 forêt 7

Carte 24×24, 1104 arêtes ; coûts 1/3/7 ; graine 20261015, RNG .NET.


## 2. L'oracle exact

Dijkstra sert de référence sur le graphe complet. Le nombre de sommets explorés est un compteur de recherche, pas un temps ni le coût total du prétraitement. L'optimum ainsi mesuré servira à auditer les bornes ; il n'est pas utilisé pour construire le bracket abstrait.

In [3]:
#nullable enable
record Recherche(Dictionary<int,double> Dist,Dictionary<int,int> Pred,int Explores);
Recherche Dijkstra(Graphe g,int source,int? cible=null){if(!g.ContainsKey(source))throw new ArgumentException("Source absente");var d=new Dictionary<int,double>{{source,0}};var pred=new Dictionary<int,int>();var vus=new HashSet<int>();var q=new PriorityQueue<int,(double,int)>();q.Enqueue(source,(0,source));while(q.Count>0){int u=q.Dequeue();if(!vus.Add(u))continue;if(cible==u)break;foreach(var(v,w)in g[u]){if(!double.IsFinite(w)||w<=0)throw new ArgumentException("Poids positif fini requis");double nd=d[u]+w;if(!d.TryGetValue(v,out double ancien)||nd<ancien){d[v]=nd;pred[v]=u;q.Enqueue(v,(nd,v));}}}return new(d,pred,vus.Count);}
int[] Chemin(Recherche r,int s,int t){if(!r.Dist.ContainsKey(t))throw new InvalidOperationException("Cible inaccessible");var path=new List<int>{t};while(path[^1]!=s)path.Add(r.Pred[path[^1]]);path.Reverse();return path.ToArray();}
var reference=Dijkstra(carte.G,depart,but);double optimal=reference.Dist[but];Console.WriteLine($"{Nom(depart,carte)} → {Nom(but,carte)} : optimum {optimal:F0}, {reference.Explores} sommets explorés sur 576.");

(0,0) → (23,23) : optimum 140, 575 sommets explorés sur 576.


## 3. Compresser en provinces

Des blocs 6×6 forment seize provinces. L'arc abstrait entre deux provinces coûte la traversée concrète **minimale** ; les déplacements internes sont oubliés. Tout trajet concret paie au moins les traversées de sa projection : le plus court trajet abstrait est donc un minorant.

Les provinces sont encodées par un entier, rangée de bloc×nombre de colonnes de blocs+colonne. Cet encodage conserve les classes géographiques.

Sur cette carte C#, l’oracle donne 140 avec 575 sommets explorés sur 576 : cette recherche exacte fournit une référence, pas un gain de compression.

In [4]:
#nullable enable
int[] Partition(Carte m,int hauteur,int largeur){if(hauteur<=0||largeur<=0||m.H%hauteur!=0||m.W%largeur!=0)throw new ArgumentException("Blocs rectangulaires divisant la carte requis");return Enumerable.Range(0,m.H*m.W).Select(u=>(u/m.W/hauteur)*(m.W/largeur)+(u%m.W/largeur)).ToArray();}
Dictionary<(int A,int B),List<Arete>> Traversees(Carte m,int[] part){if(part.Length!=m.H*m.W||part.Any(v=>v<0))throw new ArgumentException("Partition totale non négative requise");var crosses=new Dictionary<(int,int),List<Arete>>();foreach(var e in m.Aretes){int a=part[e.U],b=part[e.V];if(a==b)continue;var key=a<b?(a,b):(b,a);if(!crosses.ContainsKey(key))crosses[key]=new();crosses[key].Add(e);}return crosses;}
Graphe Abstrait(Carte m,int[] part,string mode="min"){var g=new Graphe();foreach(int a in part.Distinct())g[a]=new();foreach(var kv in Traversees(m,part)){double w=mode=="min"?kv.Value.Min(e=>e.W):mode=="moyenne"?kv.Value.Average(e=>e.W):throw new ArgumentException("Mode inconnu");g[kv.Key.A].Add((kv.Key.B,w));g[kv.Key.B].Add((kv.Key.A,w));}return g;}
(double,int[]) CheminAbstrait(Graphe g,int s,int t){if(s==t)return(0,new[]{s});var r=Dijkstra(g,s,t);return(r.Dist[t],Chemin(r,s,t));}
var part6=Partition(carte,6,6);var min6=Abstrait(carte,part6);var simple=CheminAbstrait(min6,part6[depart],part6[but]);Console.WriteLine($"k=6 : {part6.Distinct().Count()} provinces ; borne MIN {simple.Item1:F0}, {simple.Item2.Length} provinces sur le trajet abstrait.");

k=6 : 16 provinces ; borne MIN 14, 7 provinces sur le trajet abstrait.


### Auditer l'admissibilité

Deux cents requêtes longues (écart Manhattan>8) et deux cents courtes (écart≤8) confrontent le minorant à une recherche exacte. Zéro violation conforte le modèle sur ces requêtes ; le raisonnement de projection établit sa garantie générale, pas le seul échantillon.

La compression k=6 donne 16 provinces et une borne 14 pour un optimum 140 : la garantie est correcte mais faible. Sur les 400 requêtes, aucune violation MIN n’est observée.

In [5]:
#nullable enable
(int A,int B)[] Paires(Carte m,int nb,bool courtes,int graine=7){var r=new Random(graine);var list=new List<(int,int)>();while(list.Count<nb){int a=r.Next(m.H*m.W),b=r.Next(m.H*m.W);int ecart=Math.Abs(a/m.W-b/m.W)+Math.Abs(a%m.W-b%m.W);if(a!=b&&((ecart<=8)==courtes))list.Add((a,b));}return list.ToArray();}
record Violation(int A,int B,double Borne,double Exact);
List<Violation> Violations(Carte m,int[] part,Graphe g,(int A,int B)[] paires){var v=new List<Violation>();foreach(var p in paires){double exact=Dijkstra(m.G,p.A,p.B).Dist[p.B];double borne=CheminAbstrait(g,part[p.A],part[p.B]).Item1;if(borne>exact+1e-9)v.Add(new(p.A,p.B,borne,exact));}return v;}
var longues=Paires(carte,200,false);var courtes=Paires(carte,200,true);var vMinLong=Violations(carte,part6,min6,longues);var vMinCourt=Violations(carte,part6,min6,courtes);Console.WriteLine($"MIN k=6 : {vMinLong.Count}/200 violations longues, {vMinCourt.Count}/200 courtes.");if(vMinLong.Count+vMinCourt.Count!=0)throw new Exception("MIN inadmissible");

MIN k=6 : 0/200 violations longues, 0/200 courtes.


## 4. Le contre-témoin de l'agrégation moyenne

Une moyenne des traversées paraît représentative, mais peut surpayer le passage le moins cher. La dette interne oubliée peut cacher cette inflation sur les longues requêtes ; les courtes la détectent plus facilement. Une violation suffit à interdire l'usage de cette agrégation comme minorant universel.

In [6]:
#nullable enable
var part3=Partition(carte,3,3);var moyenne3=Abstrait(carte,part3,"moyenne");var vMoyCourt=Violations(carte,part3,moyenne3,courtes);var vMoyLong=Violations(carte,part3,moyenne3,longues);Console.WriteLine($"MOYENNE k=3 : {vMoyCourt.Count}/200 violations courtes, {vMoyLong.Count}/200 longues.");if(vMoyCourt.Count>0){var v=vMoyCourt[0];Console.WriteLine($"Contre-témoin : {Nom(v.A,carte)} → {Nom(v.B,carte)}, borne {v.Borne:F2} > optimum {v.Exact:F0}.");}

MOYENNE k=3 : 2/200 violations courtes, 0/200 longues.


Contre-témoin : (3,18) → (2,17), borne 8.67 > optimum 6.


### Exercice 1 — Agréger au maximum

Construisez le graphe où chaque frontière coûte la traversée la plus chère. Le contrôleur vérifie toutes les classes, adjacences et valeurs finies sur deux cartes, puis mesure les violations sur les deux cents requêtes courtes de la carte principale. Comparez le nombre et le premier contre-témoin à la moyenne.

La moyenne produit ici 2 violations sur 200 requêtes courtes et aucune sur les 200 longues. Le témoin (3,18) → (2,17) coûte 6 en concret alors que sa moyenne abstraite vaut 8,67 : une seule telle inflation réfute l’admissibilité générale.

In [7]:
#nullable enable
Graphe? AgregatMax(Carte m,int[] part){return null; /* TODO étudiant : maximum des coûts de traversée. */}
Graphe ReferenceMax(Carte m,int[] part){var g=new Graphe();foreach(int p in part.Distinct())g[p]=new();foreach(var kv in Traversees(m,part)){double w=kv.Value.Max(e=>e.W);g[kv.Key.A].Add((kv.Key.B,w));g[kv.Key.B].Add((kv.Key.A,w));}return g;}
void VerifierGraphe(Graphe proposition,Graphe attendu){if(!proposition.Keys.ToHashSet().SetEquals(attendu.Keys))throw new ArgumentException("Classes incorrectes");foreach(int a in attendu.Keys){if(proposition[a] is null||proposition[a].Count!=attendu[a].Count||proposition[a].Select(x=>x.V).Distinct().Count()!=proposition[a].Count)throw new ArgumentException("Adjacence incorrecte");foreach(var(v,w)in proposition[a])if(!double.IsFinite(w)||!attendu[a].Any(x=>x.V==v&&Math.Abs(x.W-w)<1e-10))throw new ArgumentException("Coût ou voisin incorrect");}}
var maxPropose=AgregatMax(carte,part3);var miniCarte=new Carte(2,3,new[]{1,1,7,3,1,1});var miniPart=Partition(miniCarte,1,1);var maxMini=AgregatMax(miniCarte,miniPart);if(maxPropose is not null)VerifierGraphe(maxPropose,ReferenceMax(carte,part3));if(maxMini is not null)VerifierGraphe(maxMini,ReferenceMax(miniCarte,miniPart));if((maxPropose is null)!=(maxMini is null))throw new ArgumentException("Exercice 1 : réponse partielle");if(maxPropose is null&&maxMini is null)Console.WriteLine("Exercice 1 à compléter : agrégation MAX.");else{var vmax=Violations(carte,part3,maxPropose!,courtes);Console.WriteLine($"MAX : {vmax.Count}/200 violations courtes, contre {vMoyCourt.Count} pour MOYENNE.");if(vmax.Count>0){var v=vmax[0];Console.WriteLine($"Premier contre-témoin MAX : {Nom(v.A,carte)}→{Nom(v.B,carte)}, {v.Borne:F2}>{v.Exact:F0}");}}

Exercice 1 à compléter : agrégation MAX.


## 5. Payer la première sortie, sans la refacturer aux retours

Pour quitter la province initiale, une route doit d'abord atteindre une frontière depuis sa position réelle. La distance interne minimale peut renforcer la première sortie. Une revisite ne doit cependant pas repayer ce trajet initial : elle représente un état d'information différent.

Le graphe distingue donc le départ et la copie de revisite de sa province (étiquette −1, hors des identifiants géographiques non négatifs). Les autres transitions restent au minimum. Pour une destination dans la province de départ, la borne vaut zéro. Tout trajet concret se projette dans ce graphe en payant au moins la première sortie puis chaque transition.

Le contre-témoin 8×8 compare aussi la variante volontairement non admissible qui fusionne ces deux états. Il n'est pas utilisé pour alimenter les mesures de la carte principale.

In [8]:
#nullable enable
Graphe Renforce(Carte m,int[] part,int source,bool distinguerRetours=true){var normal=Abstrait(m,part);int s=part[source],retour=distinguerRetours?-1:s;var g=new Graphe();foreach(var kv in normal){int a=kv.Key==s?retour:kv.Key;g[a]=kv.Value.Select(x=>(x.V==s?retour:x.V,x.W)).ToList();}var interne=new Graphe();foreach(int u in m.G.Keys.Where(u=>part[u]==s))interne[u]=m.G[u].Where(x=>part[x.V]==s).ToList();var d=Dijkstra(interne,source).Dist;var sorties=new Dictionary<int,double>();foreach(var e in m.Aretes){int point,dest;if(part[e.U]==s&&part[e.V]!=s){point=e.U;dest=part[e.V];}else if(part[e.V]==s&&part[e.U]!=s){point=e.V;dest=part[e.U];}else continue;if(d.TryGetValue(point,out double interneCout)){double w=interneCout+e.W;if(!sorties.TryGetValue(dest,out double actuel)||w<actuel)sorties[dest]=w;}}g[s]=sorties.Select(x=>(x.Key,x.Value)).ToList();return g;}
var renforce6=Renforce(carte,part6,depart);double borne6=CheminAbstrait(renforce6,part6[depart],part6[but]).Item1;var depuisDepart=Dijkstra(carte.G,depart).Dist;int violationsRenforce=longues.Concat(courtes).Count(p=>CheminAbstrait(renforce6,part6[depart],part6[p.B]).Item1>depuisDepart[p.B]+1e-9);Console.WriteLine($"k=6 : borne renforcée {borne6:F0}, borne MIN {simple.Item1:F0}, optimum {optimal:F0} ; {violationsRenforce}/400 violations.");if(violationsRenforce!=0)throw new Exception("Borne renforcée invalide");
var fixture8=new Carte(8,8,new[]{3,1,1,1,1,3,3,3,1,7,1,1,1,3,1,1,1,3,7,7,1,1,1,1,3,7,1,1,1,7,3,1,7,1,1,7,7,3,3,3,3,1,3,7,1,1,1,3,3,1,1,7,1,1,1,1,3,3,1,1,7,7,1,3});var fpart=Partition(fixture8,4,4);int fs=0,ft=33;double fexact=Dijkstra(fixture8.G,fs,ft).Dist[ft],faux=CheminAbstrait(Renforce(fixture8,fpart,fs,false),fpart[fs],fpart[ft]).Item1,bon=CheminAbstrait(Renforce(fixture8,fpart,fs),fpart[fs],fpart[ft]).Item1;Console.WriteLine($"Retour de province 8×8 : sans distinction {faux:F0} > optimum {fexact:F0} ; avec distinction {bon:F0} ≤ {fexact:F0}.");if(faux!=14||fexact!=13||bon!=8)throw new Exception("Contre-témoin des retours non reproduit");

k=6 : borne renforcée 45, borne MIN 14, optimum 140 ; 0/400 violations.


Retour de province 8×8 : sans distinction 14 > optimum 13 ; avec distinction 8 ≤ 13.


## 6. Relever une route concrète

Le chemin du graphe **MIN** désigne une union de provinces : un couloir. Dijkstra restreint à ce couloir donne un vrai trajet, donc un majorant de l'optimum. La borne renforcée fournit le minorant ; ces deux calculs construisent le bracket sans consulter la valeur optimale.

L'oracle intervient ensuite pour vérifier le bracket et chiffrer la dette. La carte, les traversées et la recherche interne de première sortie ont un coût ; les compteurs de sommets du couloir n'incluent pas tout ce prétraitement. Aucun gain de vitesse totale n'est déduit de ce seul compteur.

In [9]:
#nullable enable
record Releve(double Cout,int[] Trajet,int Taille,int Explores,int[] Provinces);
Releve Relever(Carte m,int[] part,Graphe abstrait,int source,int but){var provinces=CheminAbstrait(abstrait,part[source],part[but]).Item2;var ensemble=provinces.ToHashSet();var sous=new Graphe();foreach(int u in m.G.Keys.Where(u=>ensemble.Contains(part[u])))sous[u]=m.G[u].Where(x=>ensemble.Contains(part[x.V])).ToList();var r=Dijkstra(sous,source,but);return new(r.Dist[but],Chemin(r,source,but),sous.Count,r.Explores,provinces);}
var releve6=Relever(carte,part6,min6,depart,but);if(borne6>optimal||releve6.Cout<optimal)throw new Exception("Bracket invalide");Console.WriteLine($"k=6 : couloir {releve6.Taille}/576 sommets, {releve6.Explores} explorés ; bracket [{borne6:F0},{releve6.Cout:F0}], optimum {optimal:F0}, dette {releve6.Cout-optimal:F0}.");

k=6 : couloir 252/576 sommets, 251 explorés ; bracket [45,170], optimum 140, dette 30.


## 7. Une famille d'échelles

Mesurons les six tailles 12, 8, 6, 4, 3, 2. Ces partitions ne constituent pas toutes une chaîne de raffinements emboîtés : les frontières de blocs 8 et 12 diffèrent. Ni la dette ni le resserrement du bracket ne sont supposés monotones. À chaque échelle, le tableau vérifie MIN≤borne renforcée≤optimum≤relevé.

À k=6, le bracket renforcé/relevé est [45,170] pour l’optimum 140 ; le couloir contient 252 sommets et la dette vaut 30. La taille du couloir ne mesure pas tous les coûts de calcul.

In [10]:
#nullable enable
record Echelle(int K,int Provinces,double Simple,double Renforcee,Releve Releve);
var echelles=new List<Echelle>();Console.WriteLine(" k provinces borne MIN renforcée relevé dette couloir explorés");foreach(int k in new[]{12,8,6,4,3,2}){var part=Partition(carte,k,k);var g=Abstrait(carte,part);double bmin=CheminAbstrait(g,part[depart],part[but]).Item1;double br=CheminAbstrait(Renforce(carte,part,depart),part[depart],part[but]).Item1;var releve=Relever(carte,part,g,depart,but);if(bmin>br+1e-9||br>optimal+1e-9||releve.Cout<optimal)throw new Exception("Certificat d'échelle invalide");echelles.Add(new(k,part.Distinct().Count(),bmin,br,releve));Console.WriteLine($"{k,2} {part.Distinct().Count(),9} {bmin,8:F0} {br,9:F0} {releve.Cout,6:F0} {releve.Cout-optimal,5:F0} {releve.Taille,7} {releve.Explores,8}");}

 k provinces borne MIN renforcée relevé dette couloir explorés


12         4        4        57    140     0     432      431


 8         9        8        48    140     0     320      319


 6        16       14        45    170    30     252      251


 4        36       22        47    160    20     176      175


 3        64       36        54    160    20     135      133


 2       144       60        67    144     4      92       91


### Lecture graphique

La même figure représente les minorants et le trajet relevé, avec la référence exacte pointillée. L'axe des provinces est logarithmique, comme dans le Python. Les valeurs proviennent du tableau précédent ; une courbe plus fine n'achète pas automatiquement une dette plus faible.

Les dettes mesurées aux six tailles sont 0, 0, 30, 20, 20 et 4. Passer de 9 à 16 provinces augmente ici la dette de 0 à 30, sans violer le bracket. Le nombre de classes seul ne prédit donc pas le coût de la route choisie.

In [11]:
#nullable enable
var chart=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='850' height='450' viewBox='0 0 850 450'><rect width='850' height='450' fill='white'/><text x='425' y='25' text-anchor='middle'>Bornes et relevé selon l'échelle d'abstraction</text><line x1='75' y1='345' x2='800' y2='345' stroke='black'/><line x1='75' y1='55' x2='75' y2='345' stroke='black'/><text x='430' y='395' text-anchor='middle'>Nombre de provinces — échelle logarithmique</text><text transform='translate(19 210) rotate(-90)' text-anchor='middle'>Coût du trajet</text>");double ymax=echelles.Max(v=>v.Releve.Cout)*1.12;double X(int p)=>90+690*(Math.Log(p)-Math.Log(4))/(Math.Log(144)-Math.Log(4));double Y(double c)=>345-270*c/ymax;for(int t=0;t<=ymax;t+=20)chart.Append($"<line x1='75' y1='{Y(t):F2}' x2='800' y2='{Y(t):F2}' stroke='#ddd'/><text x='65' y='{Y(t)+4:F2}' text-anchor='end'>{t}</text>");foreach(var e in echelles)chart.Append($"<text x='{X(e.Provinces):F2}' y='367' text-anchor='middle'>{e.Provinces}</text>");var series=new[]{("MIN","#4477aa",echelles.Select(e=>e.Simple).ToArray()),("renforcée","#228833",echelles.Select(e=>e.Renforcee).ToArray()),("relevé","#cc8844",echelles.Select(e=>e.Releve.Cout).ToArray())};foreach(var(nom,couleur,values)in series){chart.Append($"<polyline fill='none' stroke='{couleur}' stroke-width='2' points='"+string.Join(" ",echelles.Select((e,i)=>$"{X(e.Provinces):F2},{Y(values[i]):F2}"))+"'/>");foreach(var point in echelles.Select((e,i)=>(e.Provinces,Value:values[i])))chart.Append($"<circle cx='{X(point.Provinces):F2}' cy='{Y(point.Value):F2}' r='4' fill='{couleur}'/>");}chart.Append($"<line x1='75' y1='{Y(optimal):F2}' x2='800' y2='{Y(optimal):F2}' stroke='black' stroke-dasharray='5 4'/>");string[] legendes={"borne MIN","borne renforcée","relevé","optimum"};string[] cs={"#4477aa","#228833","#cc8844","black"};for(int i=0;i<4;i++)chart.Append($"<rect x='{105+i*180}' y='420' width='13' height='13' fill='{cs[i]}'/><text x='{124+i*180}' y='431' font-size='13'>{legendes[i]}</text>");chart.Append("</svg>");display(new Dessin(chart.ToString()));Console.WriteLine("Six échelles mesurées ; ni dette monotone ni vitesse totale déduite du nombre de provinces.");

Bornes et relevé selon l'échelle d'abstraction Nombre de provinces — échelle logarithmique Coût du trajet 0 20 40 60 80 100 120 140 160 180 4 9 16 36 64 144 borne MIN borne renforcée relevé optimum

Six échelles mesurées ; ni dette monotone ni vitesse totale déduite du nombre de provinces.


## 8. Exercice 2 — Une autre géométrie à seize classes

Construisez la partition en rectangles 12 rangées × 3 colonnes, ainsi qu'une petite fixture. Utilisez l'encodage déclaré de la fonction Partition. Le contrôleur vérifie toutes les cellules, puis construit réellement l'abstraction et relève une route. Comparez coût, dette et taille de couloir aux blocs 6×6.

In [12]:
#nullable enable
int[]? PartitionRectangles(Carte m,int hauteur,int largeur){return null; /* TODO étudiant : encoder les coordonnées de blocs rectangulaires. */}
bool VerifierPartition(int[]? reponse,Carte m,int h,int w){if(reponse is null)return false;var expected=Partition(m,h,w);if(!reponse.SequenceEqual(expected))throw new ArgumentException("Partition rectangulaire incorrecte");return true;}
var rect1=PartitionRectangles(carte,12,3);var rect2=PartitionRectangles(miniCarte,1,3);bool r1=VerifierPartition(rect1,carte,12,3),r2=VerifierPartition(rect2,miniCarte,1,3);if(r1!=r2)throw new ArgumentException("Exercice 2 : réponse partielle");if(!r1&&!r2)Console.WriteLine("Exercice 2 à compléter : partition rectangulaire.");else{var g=Abstrait(carte,rect1!);var r=Relever(carte,rect1!,g,depart,but);Console.WriteLine($"Rectangles 12×3 : {rect1!.Distinct().Count()} classes, relevé {r.Cout:F0}, dette {r.Cout-optimal:F0}, couloir {r.Taille} ; blocs 6×6 : relevé {releve6.Cout:F0}, dette {releve6.Cout-optimal:F0}, couloir {releve6.Taille}.");}

Exercice 2 à compléter : partition rectangulaire.


## 9. Exercice 3 — Raffiner là où la route paie le plus

Choisissez une province maximisant les coûts **strictement internes** de la route relevée à k=12. Les arêtes traversant une frontière ne sont pas attribuées à l'intérieur d'une province. Les gardes vérifient deux dictionnaires discriminants et le vrai trajet ; les ex aequo maximaux sont admis. Le host effectue ensuite la coupure en deux moitiés, reconstruit le graphe et mesure la nouvelle dette.

In [13]:
#nullable enable
Dictionary<int,double> DepensesInternes(Carte m,int[] part,int[] trajet){var costs=new Dictionary<int,double>();for(int i=0;i+1<trajet.Length;i++){int u=trajet[i],v=trajet[i+1];if(part[u]!=part[v])continue;double w=m.G[u].Single(e=>e.V==v).W;costs[part[u]]=costs.GetValueOrDefault(part[u])+w;}return costs;}
int? ChoixProvince(Dictionary<int,double> depenses){return null; /* TODO étudiant : une clé atteignant le maximum. */}
bool VerifierChoix(int? choix,Dictionary<int,double> costs){if(choix is null)return false;if(!costs.ContainsKey(choix.Value)||!double.IsFinite(costs[choix.Value])||costs[choix.Value]!=costs.Values.Max())throw new ArgumentException("Province choisie non maximale");return true;}
var part12=Partition(carte,12,12);var lift12=Relever(carte,part12,Abstrait(carte,part12),depart,but);var depenses=DepensesInternes(carte,part12,lift12.Trajet);Console.WriteLine($"Avant coupure : relevé {lift12.Cout:F0}, dette {lift12.Cout-optimal:F0}, couloir {lift12.Taille} ; coûts strictement internes : "+string.Join(" ; ",depenses.OrderByDescending(x=>x.Value).Select(x=>$"province {x.Key}:{x.Value:F0}")));
var costsA=new Dictionary<int,double>{{0,1},{1,4}};var costsB=new Dictionary<int,double>{{9,2},{7,4}};bool c1=VerifierChoix(ChoixProvince(costsA),costsA),c2=VerifierChoix(ChoixProvince(costsB),costsB);int? choix=ChoixProvince(depenses);bool c3=VerifierChoix(choix,depenses);if((c1||c2||c3)&&!(c1&&c2&&c3))throw new ArgumentException("Exercice 3 : réponse partielle");if(!c1&&!c2&&!c3)Console.WriteLine("Exercice 3 à compléter : province à raffiner.");else{var nouvelle=part12.ToArray();int midpoint=(choix!.Value/2)*12+6;int premier=part12.Max()+1;for(int u=0;u<nouvelle.Length;u++)if(part12[u]==choix.Value)nouvelle[u]=premier+(u/carte.W<midpoint?0:1);var apres=Relever(carte,nouvelle,Abstrait(carte,nouvelle),depart,but);Console.WriteLine($"Après coupure de {choix} : {nouvelle.Distinct().Count()} classes, relevé {apres.Cout:F0}, dette {apres.Cout-optimal:F0}, couloir {apres.Taille}.");}

Avant coupure : relevé 140, dette 0, couloir 432 ; coûts strictement internes : province 0:61 ; province 3:57 ; province 1:18


Exercice 3 à compléter : province à raffiner.


## À retenir

|Geste|Preuve disponible|Limite|
|---|---|---|
|MIN|projection minorante et audits 400 requêtes|la borne peut être très faible|
|MOYENNE/MAX|contre-témoins mesurables|pas des minorants universels|
|Première sortie/revisite|états distincts et contre-témoin 14 > 13|borne ancrée à une source|
|Relèvement|route concrète vérifiable|dette à mesurer contre l'oracle|
|Plusieurs échelles|brackets vérifiés ligne par ligne|pas de monotonie ni de vitesse totale prouvée|

## Vérifiez votre compréhension

1. Pourquoi l'agrégation minimale reste-t-elle admissible ?
2. Pourquoi faut-il distinguer départ et revisite de la même province ?
3. Un couloir plus petit garantit-il une route moins endettée ?

<details><summary>Réponses</summary>

1. Toute route concrète paie au moins le minimum de chaque frontière traversée ; oublier l'interne ne peut augmenter la borne.
2. La première sortie doit payer le trajet interne depuis la source, mais une route revenue d'ailleurs n'a pas à le repayer. La fusion de ces deux états produit le contre-témoin 14 > 13.
3. Non. Le couloir choisi par le minimum abstrait peut omettre une bonne route ; le nombre de provinces ne détermine pas la dette. Le prétraitement doit aussi être compté avant de comparer des coûts de calcul.

</details>

## Pour aller plus loin

- [Frontieres-03 — MAPF](Frontieres-03-MAPF-Guarantee-Audit-Python.ipynb) : valider indépendamment un trajet.
- [Metajeux-03](../../GameTheory/Metajeux/Metajeux-03-Abstraction-a-Dette-Python.ipynb) : abstraction et dette sur un autre terrain.
- Culberson, J. C. & Schaeffer, J. (1998), *Pattern Databases*, Computational Intelligence 14(3).
- Holte, R. C., Grahn, H., Hansen, E. A. & Zilberstein, S. (2005), *Maximizing over Multiple Abstractions for Heuristic Search*, SARA 2005.

Les expériences portent sur des cartes finies sans obstacle infranchissable. Elles ne constituent ni un benchmark de planification à grande échelle ni une preuve de correction de tout programme étudiant.